# Langbridge SDK Local Runtime Notebook

This notebook uses `LangbridgeClient.local(config_path=...)` against the local runtime defined in `langbridge_config.yml`.

In [1]:
from pathlib import Path
import sys

EXAMPLE_DIR = Path.cwd()
REPO_ROOT = EXAMPLE_DIR.parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [2]:
import dotenv
dotenv.load_dotenv(".env")

True

In [3]:
import pandas as pd
from langbridge import LangbridgeClient
from setup import setup_database

CONFIG_PATH = EXAMPLE_DIR / "langbridge_config.yml"
setup_database()
client = LangbridgeClient.local(config_path=str(CONFIG_PATH))

/home/callumwhi/langbridgedev/langbridge/langbridge/packages/common/langbridge_common/contracts/datasets.py:217: UserWarning: Field name "schema" in "DatasetSelectionRequest" shadows an attribute in parent "_Base"
  class DatasetSelectionRequest(_Base):
/home/callumwhi/langbridgedev/langbridge/langbridge/packages/common/langbridge_common/contracts/datasets.py:230: UserWarning: Field name "schema" in "DatasetEnsureRequest" shadows an attribute in parent "_Base"
  class DatasetEnsureRequest(_Base):
/home/callumwhi/langbridgedev/langbridge/langbridge/packages/runtime/models/metadata.py:32: UserWarning: Field name "schema" in "ConnectionMetadata" shadows an attribute in parent "RuntimeModel"
  class ConnectionMetadata(RuntimeModel):
/home/callumwhi/langbridgedev/langbridge/langbridge/packages/runtime/models/jobs.py:139: UserWarning: Field name "schema" in "DatasetSelectionRequest" shadows an attribute in parent "RuntimeRequestModel"
  class DatasetSelectionRequest(RuntimeRequestModel):
/ho

## List datasets

In [4]:
datasets = await client.datasets.list()
pd.DataFrame([item.model_dump(mode="json") for item in datasets.items])

,id,name,label,description,connector,semantic_model
0,59c272b8-4234-53a1-b2d8-b68230d815e5,shopify_orders,Shopify Orders,Order-level commerce performance data with rev...,commerce_demo,commerce_performance
1,3cda9df7-089b-5190-b836-8965d3e956d2,customers,Customers,Customer profile and segmentation data.,commerce_demo,commerce_performance
2,0f1fc07e-47ca-5b1c-80cf-437052c1da6c,support_tickets,Support Tickets,Support cases tied to customer and order activ...,commerce_demo,commerce_performance


## Run a semantic query


In [5]:
country_sales = await client.semantic.query(
    "commerce_performance",
    measures=["shopify_orders.net_sales", "shopify_orders.gross_margin"],
    dimensions=["shopify_orders.country"],
    order={"shopify_orders.net_sales": "desc"},
    limit=5,
)
print(pd.DataFrame(country_sales.rows))


  shopify_orders.country  shopify_orders.net_sales  \
0                 Canada                  18064.42   
1            Netherlands                  16766.80   
2          United States                   8741.73   

   shopify_orders.gross_margin  
0                      8813.47  
1                      8099.00  
2                      4533.28  


## Run SQL directly

In [6]:
sql_result = client.sql.query(
    query="""
    SELECT acquisition_channel, ROUND(SUM(net_revenue), 2) AS net_sales
    FROM orders_enriched
    GROUP BY acquisition_channel
    ORDER BY net_sales DESC
    LIMIT 5
    """,
)
pd.DataFrame(sql_result.rows)

,acquisition_channel,net_sales
0,Organic Search,8606.91
1,Paid Search,7234.94
2,Email,6805.70
3,Partner,6289.38
4,Paid Social,5762.43


## Ask the configured local agent

In [ ]:
answer = await client.agents.ask("Show me top countries by net sales this quarter")
print(answer.text)
print(pd.DataFrame(answer.result["rows"]))